**Data Collection & Preprocessing**

I will start by fetching stock market data from Yahoo Finance, then clean and process it for training our risk management model.

**Install Required Libraries**

In [1]:
pip install finrl yfinance stable-baselines3 gym pandas numpy matplotlib seaborn


  Cloning https://github.com/AI4Finance-Foundation/ElegantRL.git to /tmp/pip-install-h2porspk/elegantrl_d99bbe57d0ea42e796c1373aabf81af2
  Running command git clone --filter=blob:none --quiet https://github.com/AI4Finance-Foundation/ElegantRL.git /tmp/pip-install-h2porspk/elegantrl_d99bbe57d0ea42e796c1373aabf81af2
  Resolved https://github.com/AI4Finance-Foundation/ElegantRL.git to commit 2fa34dd9236498beada8d8443d927970a9de1f7f
  Preparing metadata (setup.py) ... done


In [2]:
!pip install stockstats

In [3]:
!pip install finrl --upgrade

  Using cached FinRL-0.3.7-py3-none-any.whl.metadata (909 bytes)
Using cached FinRL-0.3.7-py3-none-any.whl (127 kB)
  Attempting uninstall: finrl
    Found existing installation: finrl 0.3.6
    Uninstalling finrl-0.3.6:
      Successfully uninstalled finrl-0.3.6


In [4]:
!pip install alpaca-trade-api

In [5]:
!pip install git+https://github.com/AI4Finance-LLC/FinRL-Library.git

  Cloning https://github.com/AI4Finance-LLC/FinRL-Library.git to /tmp/pip-req-build-44q_t4t5
  Running command git clone --filter=blob:none --quiet https://github.com/AI4Finance-LLC/FinRL-Library.git /tmp/pip-req-build-44q_t4t5
  Resolved https://github.com/AI4Finance-LLC/FinRL-Library.git to commit 3a10805389c07e13e94333e94dcfe9beafce49b0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/AI4Finance-Foundation/ElegantRL.git to /tmp/pip-install-7lrccxea/elegantrl_c635dd6fa83d4da28ef1745b8ec7c867
  Running command git clone --filter=blob:none --quiet https://github.com/AI4Finance-Foundation/ElegantRL.git /tmp/pip-install-7lrccxea/elegantrl_c635dd6fa83d4da28ef1745b8ec7c867
  Resolved https://github.com/AI4Finance-Foundation/ElegantRL.git to commit 2fa34dd9236498beada8d8443d927970a9de1f7f
  Preparing metadata (setup.py) ... done
  Created wheel for finrl: filename=finrl-0.3.6-p

In [6]:
!pip install exchange-calendars

In [7]:
!pip install wrds

In [8]:
!pip install finrl


  Cloning https://github.com/AI4Finance-Foundation/ElegantRL.git to /tmp/pip-install-9aqoc4tw/elegantrl_3492c341fe5d48c2874122695428f59b
  Running command git clone --filter=blob:none --quiet https://github.com/AI4Finance-Foundation/ElegantRL.git /tmp/pip-install-9aqoc4tw/elegantrl_3492c341fe5d48c2874122695428f59b
  Resolved https://github.com/AI4Finance-Foundation/ElegantRL.git to commit 2fa34dd9236498beada8d8443d927970a9de1f7f
  Preparing metadata (setup.py) ... done


**Import Libraries**

In [9]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns

# FinRL Libraries
from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.meta.preprocessor.preprocessors import FeatureEngineer
from finrl.config import INDICATORS


**Manually Fetch Stock Data**

In [10]:
# Define date range
start_date = "2023-01-01"
end_date = "2023-12-31"

# List of stock tickers
ticker_list = ["AAPL", "GOOGL", "MSFT", "AMZN", "JPM", "TSLA"]

# Define the required column format
expected_columns = ["Date", "Open", "High", "Low", "Close", "Adj Close", "Volume", "Ticker"]

# Initialize an empty DataFrame
df_list = []

# Loop through each ticker and fetch data separately
for ticker in ticker_list:
    try:
        stock_data = yf.download(ticker, start=start_date, end=end_date)
        stock_data["Ticker"] = ticker  # Add a column for the ticker symbol
        stock_data.reset_index(inplace=True)  #  date is a column
        df_list.append(stock_data)
        print(f" Successfully fetched data for {ticker}")
    except Exception as e:
        print(f" Error fetching data for {ticker}: {e}")

# Combine all stock data into one DataFrame
df = pd.concat(df_list, ignore_index=True)

# Print the actual column names in the DataFrame for debugging
print("Actual DataFrame columns:", df.columns)

# Ensure column order is consistent
# If 'Adj Close' is not found, try 'Adj_Close' or check for other variations
df = df[[c for c in expected_columns if c in df.columns]] # Select only columns present in the DataFrame


# Drop missing values (if any)
df.dropna(inplace=True)

# Display final dataset
print(df.head())

/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


 Successfully fetched data for AAPL
 Successfully fetched data for GOOGL


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


 Successfully fetched data for MSFT
 Successfully fetched data for AMZN


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

 Successfully fetched data for JPM
 Successfully fetched data for TSLA
Actual DataFrame columns: MultiIndex([(  'Date',      ''),
            ( 'Close',  'AAPL'),
            (  'High',  'AAPL'),
            (   'Low',  'AAPL'),
            (  'Open',  'AAPL'),
            ('Volume',  'AAPL'),
            ('Ticker',      ''),
            ( 'Close', 'GOOGL'),
            (  'High', 'GOOGL'),
            (   'Low', 'GOOGL'),
            (  'Open', 'GOOGL'),
            ('Volume', 'GOOGL'),
            ( 'Close',  'MSFT'),
            (  'High',  'MSFT'),
            (   'Low',  'MSFT'),
            (  'Open',  'MSFT'),
            ('Volume',  'MSFT'),
            ( 'Close',  'AMZN'),
            (  'High',  'AMZN'),
            (   'Low',  'AMZN'),
            (  'Open',  'AMZN'),
            ('Volume',  'AMZN'),
            ( 'Close',   'JPM'),
            (  'High',   'JPM'),
            (   'Low',   'JPM'),
            (  'Open',   'JPM'),
            ('Volume',   'JPM'),
            

**Flatten MultiIndex and Standardize Columns**

In [11]:
# Fetch all tickers at once
stock_data = yf.download(ticker_list, start=start_date, end=end_date, group_by='ticker')

# Flatten MultiIndex and reformat DataFrame
flattened_data = []

for ticker in ticker_list:
    temp_df = stock_data[ticker].copy()
    temp_df["Ticker"] = ticker  # Add a ticker column
    temp_df.reset_index(inplace=True)  # Make date a column
    flattened_data.append(temp_df)

# Combine all individual DataFrames
df = pd.concat(flattened_data, ignore_index=True)

# Rename columns for consistency
df.rename(columns={"Adj Close": "Adj_Close"}, inplace=True)

# Display the first few rows
print(df.head())


/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
[*********************100%***********************]  6 of 6 completed


Price       Date        Open        High         Low       Close     Volume  \
0     2023-01-03  128.924229  129.537772  122.877812  123.768448  112117500   
1     2023-01-04  125.569527  127.321112  123.778365  125.045044   89113600   
2     2023-01-05  125.807022  126.440361  123.461690  123.718979   80962700   
3     2023-01-06  124.698663  128.934113  123.590315  128.271088   87754700   
4     2023-01-09  129.112286  132.021694  128.538320  128.795609   70790800   

Price Ticker  
0       AAPL  
1       AAPL  
2       AAPL  
3       AAPL  
4       AAPL  


**Preprocessing**

In [12]:
# Load your stock data (assuming df is already fetched)
df = df.dropna()  # Drop missing values (alternative: df.fillna(method='ffill'))

# Convert Date to datetime format
df['Date'] = pd.to_datetime(df['Date'])

# Sort data for consistency
df = df.sort_values(by=['Ticker', 'Date']).reset_index(drop=True)

# Function to calculate Moving Averages
def moving_average(df, window=10):
    df[f'MA_{window}'] = df.groupby('Ticker')['Close'].transform(lambda x: x.rolling(window=window).mean())

# Apply Moving Averages
moving_average(df, window=10)
moving_average(df, window=50)

# Calculate Relative Strength Index (RSI)
def compute_rsi(df, window=14):
    delta = df.groupby('Ticker')['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    df[f'RSI_{window}'] = 100 - (100 / (1 + rs))

compute_rsi(df, window=14)

# Calculate MACD (Moving Average Convergence Divergence)
def compute_macd(df):
    short_ema = df.groupby('Ticker')['Close'].transform(lambda x: x.ewm(span=12, adjust=False).mean())
    long_ema = df.groupby('Ticker')['Close'].transform(lambda x: x.ewm(span=26, adjust=False).mean())
    df['MACD'] = short_ema - long_ema
    df['MACD_Signal'] = df.groupby('Ticker')['MACD'].transform(lambda x: x.ewm(span=9, adjust=False).mean())

compute_macd(df)

# Calculate Bollinger Bands
def compute_bollinger_bands(df, window=20):
    rolling_mean = df.groupby('Ticker')['Close'].transform(lambda x: x.rolling(window=window).mean())
    rolling_std = df.groupby('Ticker')['Close'].transform(lambda x: x.rolling(window=window).std())
    df['BB_Upper'] = rolling_mean + (rolling_std * 2)
    df['BB_Lower'] = rolling_mean - (rolling_std * 2)

moving_average(df, window=10)
moving_average(df, window=50)
compute_rsi(df, window=14)
compute_macd(df)

compute_bollinger_bands(df, window=20)

# Normalize price values for better performance in models
df['Close_Norm'] = df.groupby('Ticker')['Close'].transform(lambda x: (x - x.min()) / (x.max() - x.min()))

# Save preprocessed data
df.to_csv('preprocessed_stock_data.csv', index=False)

# Display first few rows
print(df.head())


/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


Price       Date        Open        High         Low       Close     Volume  \
0     2023-01-03  128.924229  129.537772  122.877812  123.768448  112117500   
1     2023-01-04  125.569527  127.321112  123.778365  125.045044   89113600   
2     2023-01-05  125.807022  126.440361  123.461690  123.718979   80962700   
3     2023-01-06  124.698663  128.934113  123.590315  128.271088   87754700   
4     2023-01-09  129.112286  132.021694  128.538320  128.795609   70790800   

Price Ticker  MA_10  MA_50  RSI_14      MACD  MACD_Signal  BB_Upper  BB_Lower  \
0       AAPL    NaN    NaN     NaN  0.000000     0.000000       NaN       NaN   
1       AAPL    NaN    NaN     NaN  0.101837     0.020367       NaN       NaN   
2       AAPL    NaN    NaN     NaN  0.074680     0.031230       NaN       NaN   
3       AAPL    NaN    NaN     NaN  0.415683     0.108121       NaN       NaN   
4       AAPL    NaN    NaN     NaN  0.719956     0.230488       NaN       NaN   

Price  Close_Norm  
0        0.000674 

In [13]:
# Remove initial rows with NaN values from moving averages, RSI, and Bollinger Bands
df = df.dropna().reset_index(drop=True)


/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [14]:
# Display first few rows
print(df.head())

Price       Date        Open        High         Low       Close    Volume  \
0     2023-03-15  149.845113  151.886786  148.586406  151.629105  77167900   
1     2023-03-16  150.806460  155.068213  150.291082  154.463638  76161100   
2     2023-03-17  154.691580  155.345712  152.907589  153.621185  98944600   
3     2023-03-20  153.690557  156.416094  152.778728  155.999817  73641400   
4     2023-03-21  155.920572  157.982056  155.147497  157.863129  73938300   

Price Ticker       MA_10       MA_50     RSI_14      MACD  MACD_Signal  \
0       AAPL  149.694435  143.391291  56.553544  1.693255     1.876622   
1       AAPL  150.679593  144.005195  66.582016  1.962011     1.893700   
2       AAPL  151.073062  144.576718  63.014674  2.083011     1.931562   
3       AAPL  151.426886  145.222334  67.170790  2.343823     2.014014   
4       AAPL  152.188055  145.814175  74.194698  2.670092     2.145230   

Price    BB_Upper    BB_Lower  Close_Norm  
0      154.652717  143.284179    0.380116 

In [15]:
from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv
import pandas as pd


/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


**Implement Risk Management Model (With PPO)**

In [16]:
# Fetch VIX data
vix_data = yf.download("^VIX", start=start_date, end=end_date)

# Flatten MultiIndex in vix_data if present
if isinstance(vix_data.columns, pd.MultiIndex):
    vix_data.columns = [col[0] for col in vix_data.columns]  # Keep only first level

# Ensure 'date' column exists and is properly formatted
# Check if 'level_0' already exists as a column, if so, drop it before reset_index
if 'level_0' in vix_data.columns:
    vix_data = vix_data.drop(columns=['level_0'])
vix_data = vix_data.reset_index() # Reset the index after potentially dropping 'level_0'
vix_data.rename(columns={"Date": "date"}, inplace=True)
vix_data["date"] = pd.to_datetime(vix_data["date"])

# Ensure df["Date"] is also formatted correctly
# Changed 'date' to 'Date' to match the existing column name
df["Date"] = pd.to_datetime(df["Date"])

# Debugging: Print column names before merging
print("df columns:", df.columns)
print("vix_data columns:", vix_data.columns)

# Merge VIX data with stock data using 'Date' from df and 'date' from vix_data
df = pd.merge(df, vix_data, left_on="Date", right_on="date", how="left")  # Changed here

print(" Merge successful!")

[*********************100%***********************]  1 of 1 completed

df columns: Index(['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Ticker', 'MA_10',
       'MA_50', 'RSI_14', 'MACD', 'MACD_Signal', 'BB_Upper', 'BB_Lower',
       'Close_Norm'],
      dtype='object', name='Price')
vix_data columns: Index(['date', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='object')
 Merge successful!


In [17]:
%whos


/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


Variable                  Type         Data/Info
------------------------------------------------
FeatureEngineer           type         <class 'finrl.meta.prepro<...>cessors.FeatureEngineer'>
INDICATORS                list         n=8
StockTradingEnv           type         <class 'finrl.meta.env_st<...>trading.StockTradingEnv'>
YahooDownloader           type         <class 'finrl.meta.prepro<...>nloader.YahooDownloader'>
compute_bollinger_bands   function     <function compute_bolling<...>_bands at 0x7bed3c658e00>
compute_macd              function     <function compute_macd at 0x7bed3c659a80>
compute_rsi               function     <function compute_rsi at 0x7bed3c659940>
df                        DataFrame               Date      Open<...>n[1206 rows x 21 columns]
df_list                   list         n=6
end_date                  str          2023-12-31
expected_columns          list         n=8
flattened_data            list         n=6
moving_average            function     <func

In [18]:
df_vix = vix_data.copy()


/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [19]:
print("df_vix columns before processing:", df_vix.columns)


df_vix columns before processing: Index(['date', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='object')


In [20]:
print("df_vix columns:", df_vix.columns)


df_vix columns: Index(['date', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='object')


In [21]:
# vix_data columns: Index(['level_0', 'index', 'date', 'vix'], dtype='object')
# This line is not Python code, so no syntax correction is needed.  It appears to be output from a Python program.
pass

In [22]:
print("df_vix columns after renaming:", df_vix.columns)


df_vix columns after renaming: Index(['date', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='object')


In [23]:
df_vix = df_vix[["date", "Close"]]  # Ensure only required columns
df_vix.rename(columns={"Close": "vix"}, inplace=True)  # Rename 'Close' to 'vix'
df_vix.drop_duplicates(inplace=True)
df_vix.dropna(inplace=True)

# Merge with main dataset
df = df.merge(df_vix, on="date", how="left")

# Fill missing values
df["vix"] = df["vix"].ffill()

print(" Successfully merged VIX data!")

 Successfully merged VIX data!


In [24]:
import numpy as np
import pandas as pd
import gym
from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv
from finrl.agents.stablebaselines3.models import DRLAgent
from stable_baselines3 import PPO
from stable_baselines3.common.logger import configure
import torch


/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [25]:
def risk_adjusted_reward(trade_result, cash, portfolio_value):
    """
    Custom risk-adjusted reward function.
    - Applies penalties for high volatility, drawdowns, and risk factors.
    """
    reward = trade_result

    # Apply Stop-Loss Penalty (if losses exceed threshold, punish heavily)
    if trade_result < -0.02 * portfolio_value:  # 2% loss
        reward -= abs(trade_result) * 1.5

    # Apply Take-Profit Bonus (if gains exceed threshold, reward)
    if trade_result > 0.02 * portfolio_value:  # 2% gain
        reward += abs(trade_result) * 0.5

    # Apply Volatility Penalty (high VIX = higher risk)
    vix_penalty = df["vix"].iloc[-1] / 100  # Normalize VIX
    reward -= vix_penalty

    return reward


In [26]:
class StockTradingRiskEnv(StockTradingEnv):
    def __init__(self, df, **kwargs):
        super().__init__(df, **kwargs)

    def _calculate_reward(self, actions):
        """
        Override reward calculation with risk-adjusted function.
        """
        trade_result = self.account_information["portfolio_value"][-1] - self.account_information["portfolio_value"][-2]
        portfolio_value = self.account_information["portfolio_value"][-1]
        cash = self.account_information["cash"][-1]
        return risk_adjusted_reward(trade_result, cash, portfolio_value)


**StockTradingRiskEnv**


Risk-Adjusted Trading Environment

In [27]:
class StockTradingRiskEnv(StockTradingEnv):
    def __init__(self, df, stock_dim, hmax, initial_amount, num_stock_shares,
                 buy_cost_pct, sell_cost_pct, reward_scaling, state_space,
                 action_space, tech_indicator_list):
        super().__init__(df=df, stock_dim=stock_dim, hmax=hmax,
                         initial_amount=initial_amount, num_stock_shares=num_stock_shares,
                         buy_cost_pct=buy_cost_pct, sell_cost_pct=sell_cost_pct,
                         reward_scaling=reward_scaling, state_space=state_space,
                         action_space=action_space, tech_indicator_list=tech_indicator_list)

    def _calculate_reward(self, actions):
        """
        Override reward calculation with risk-adjusted function.
        """
        trade_result = self.account_information["portfolio_value"][-1] - self.account_information["portfolio_value"][-2]
        portfolio_value = self.account_information["portfolio_value"][-1]
        cash = self.account_information["cash"][-1]
        return risk_adjusted_reward(trade_result, cash, portfolio_value)


In [28]:
# Instead of directly accessing df["Close"], check if the column exists first:
if "Close" in df.columns and isinstance(df["Close"], np.float64):
    df["Close"] = pd.Series([df["Close"]])

# Alternatively, if you expect the column to be named 'close' (lowercase):
if "close" in df.columns and isinstance(df["close"], np.float64):
    df["close"] = pd.Series([df["close"]])

In [29]:
print(df.head())  # Check DataFrame structure
print(df.dtypes)  # Ensure 'close' is a float and a Pandas Series


        Date      Open_x      High_x       Low_x     Close_x  Volume_x Ticker  \
0 2023-03-15  149.845113  151.886786  148.586406  151.629105  77167900   AAPL   
1 2023-03-16  150.806460  155.068213  150.291082  154.463638  76161100   AAPL   
2 2023-03-17  154.691580  155.345712  152.907589  153.621185  98944600   AAPL   
3 2023-03-20  153.690557  156.416094  152.778728  155.999817  73641400   AAPL   
4 2023-03-21  155.920572  157.982056  155.147497  157.863129  73938300   AAPL   

        MA_10       MA_50     RSI_14  ...    BB_Upper    BB_Lower  Close_Norm  \
0  149.694435  143.391291  56.553544  ...  154.652717  143.284179    0.380116   
1  150.679593  144.005195  66.582016  ...  154.777357  143.211078    0.418721   
2  151.073062  144.576718  63.014674  ...  155.022553  143.093735    0.407247   
3  151.426886  145.222334  67.170790  ...  155.970522  142.626449    0.439642   
4  152.188055  145.814175  74.194698  ...  157.435610  142.231755    0.465019   

        date    Close_y   

In [30]:
df = df.reset_index(drop=True)  # Ensure it's a clean DataFrame


/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [31]:
print("Columns in df:", df.columns)  # Check column names
print("First few rows:\n", df.head())  # Check if 'close' is present


Columns in df: Index(['Date', 'Open_x', 'High_x', 'Low_x', 'Close_x', 'Volume_x', 'Ticker',
       'MA_10', 'MA_50', 'RSI_14', 'MACD', 'MACD_Signal', 'BB_Upper',
       'BB_Lower', 'Close_Norm', 'date', 'Close_y', 'High_y', 'Low_y',
       'Open_y', 'Volume_y', 'vix'],
      dtype='object')
First few rows:
         Date      Open_x      High_x       Low_x     Close_x  Volume_x Ticker  \
0 2023-03-15  149.845113  151.886786  148.586406  151.629105  77167900   AAPL   
1 2023-03-16  150.806460  155.068213  150.291082  154.463638  76161100   AAPL   
2 2023-03-17  154.691580  155.345712  152.907589  153.621185  98944600   AAPL   
3 2023-03-20  153.690557  156.416094  152.778728  155.999817  73641400   AAPL   
4 2023-03-21  155.920572  157.982056  155.147497  157.863129  73938300   AAPL   

        MA_10       MA_50     RSI_14  ...    BB_Upper    BB_Lower  Close_Norm  \
0  149.694435  143.391291  56.553544  ...  154.652717  143.284179    0.380116   
1  150.679593  144.005195  66.582016  ... 

In [32]:
# List all column names to identify potential issues
print(df.columns.tolist())


['Date', 'Open_x', 'High_x', 'Low_x', 'Close_x', 'Volume_x', 'Ticker', 'MA_10', 'MA_50', 'RSI_14', 'MACD', 'MACD_Signal', 'BB_Upper', 'BB_Lower', 'Close_Norm', 'date', 'Close_y', 'High_y', 'Low_y', 'Open_y', 'Volume_y', 'vix']


/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [33]:
df.rename(columns={"Close": "close"}, inplace=True)


In [34]:
if df.empty:
    print(" Warning: df is empty! Check data loading or merging steps.")


In [35]:
print(df_vix.head())  # Check if 'close' exists in df_vix


        date        vix
0 2023-01-03  22.900000
1 2023-01-04  22.010000
2 2023-01-05  22.459999
3 2023-01-06  21.129999
4 2023-01-09  21.969999


In [36]:
if "close" not in df_vix.columns:
    print("'close' column missing from df_vix. Available columns:", df_vix.columns)


'close' column missing from df_vix. Available columns: Index(['date', 'vix'], dtype='object')


/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


**state representation for the stock trading environment**

In [37]:
# Replace the `_initiate_state` method in StockTradingRiskEnv:
def _initiate_state(self):
    if isinstance(self.data.close, np.float64):
        state = (
            [self.initial_amount]
            + [self.data.close]  # Treat 'close' as a single value
            + self.num_stock_shares
            + sum(
                [
                    self.data[tech].values.tolist()
                    for tech in self.tech_indicator_list
                ],
                [],
            )
        )
        return state

    # Original logic for when 'close' is a Series
    else:
        state = (
            [self.initial_amount]
            + self.data.close.values.tolist()
            + self.num_stock_shares
            + sum(
                [
                    self.data[tech].values.tolist()
                    for tech in self.tech_indicator_list
                ],
                [],
            )
        )
        return state

In [38]:
# Ensure the 'Close' column is renamed to 'close' and the change is applied
df.rename(columns={"Close": "close"}, inplace=True)

# Now access the 'close' column
print(" Dataframe Type:", type(df))
# Check if 'close' column exists before accessing it
if 'close' in df.columns:
    print(" 'close' Column Type:", type(df["close"]))
    print("First 5 Rows of df:\n", df.head())

 Dataframe Type: <class 'pandas.core.frame.DataFrame'>


In [39]:
# Ensure the 'Close' column is renamed to 'close'
df.rename(columns={"Close": "close"}, inplace=True)

# Verify the column names after renaming
print(df.columns)

# Now you can proceed with your code using df["close"]
if 'close' in df.columns:  # Check if 'close' column exists before proceeding
    df["close"] = df["close"].astype(float)  # Ensure it's float
    df["close"] = df["close"].apply(lambda x: float(x))  # Explicit conversion
    df = df.copy()  # Ensure df is not a view of another dataframe

    print(" Verified 'close' column type:", type(df["close"]))

Index(['Date', 'Open_x', 'High_x', 'Low_x', 'Close_x', 'Volume_x', 'Ticker',
       'MA_10', 'MA_50', 'RSI_14', 'MACD', 'MACD_Signal', 'BB_Upper',
       'BB_Lower', 'Close_Norm', 'date', 'Close_y', 'High_y', 'Low_y',
       'Open_y', 'Volume_y', 'vix'],
      dtype='object')


In [40]:
class StockTradingRiskEnv(StockTradingEnv):
    def __init__(self, df, **kwargs):
        super().__init__(df, **kwargs)
        self.data["close"] = pd.to_numeric(self.data["close"], errors="coerce")  # Convert inside class


In [41]:
print(" DataFrame Structure Check:")
print(type(df))  # Should be <class 'pandas.DataFrame'>
print(df.dtypes)  # Ensure 'close' is float or int
print(df.head())  # Preview the data


 DataFrame Structure Check:
<class 'pandas.core.frame.DataFrame'>
Date           datetime64[ns]
Open_x                float64
High_x                float64
Low_x                 float64
Close_x               float64
Volume_x                int64
Ticker                 object
MA_10                 float64
MA_50                 float64
RSI_14                float64
MACD                  float64
MACD_Signal           float64
BB_Upper              float64
BB_Lower              float64
Close_Norm            float64
date           datetime64[ns]
Close_y               float64
High_y                float64
Low_y                 float64
Open_y                float64
Volume_y                int64
vix                   float64
dtype: object
        Date      Open_x      High_x       Low_x     Close_x  Volume_x Ticker  \
0 2023-03-15  149.845113  151.886786  148.586406  151.629105  77167900   AAPL   
1 2023-03-16  150.806460  155.068213  150.291082  154.463638  76161100   AAPL   
2 2023-03-17  154

In [42]:
import numpy as np
import gym
from gym import spaces

class StockTradingRiskEnv(gym.Env):
    """Stock Trading Environment with Risk Management"""

    def __init__(self, df, stock_dim, hmax, initial_amount, num_stock_shares,
                 buy_cost_pct, sell_cost_pct, reward_scaling,
                 state_space, action_space, tech_indicator_list):
        super(StockTradingRiskEnv, self).__init__()

        # Initialize stock data
        self.df = df
        self.stock_dim = stock_dim
        self.hmax = hmax
        self.initial_amount = initial_amount
        self.current_cash = initial_amount
        self.num_stock_shares = np.array(num_stock_shares)
        self.buy_cost_pct = buy_cost_pct
        self.sell_cost_pct = sell_cost_pct
        self.reward_scaling = reward_scaling
        self.tech_indicator_list = tech_indicator_list
        self.day = 0

        # Define action and observation space
        self.action_space = spaces.Box(low=-1, high=1, shape=(action_space,))
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(state_space,))

        # Load initial state
        self.data = self.df.iloc[self.day]

    def reset(self):
        """Resets the environment"""
        self.day = 0
        self.current_cash = self.initial_amount
        self.num_stock_shares = np.zeros(self.stock_dim)
        self.data = self.df.iloc[self.day]
        return self._get_observation()

    def step(self, action):
        """Executes an action (buy/sell/hold) and updates the environment"""
        self._trade(action)
        self.day += 1
        done = self.day >= len(self.df) - 1
        self.data = self.df.iloc[self.day] if not done else self.data
        reward = self._calculate_reward()
        return self._get_observation(), reward, done, {}

    def _trade(self, action):
        """Processes buy/sell actions"""
        for i in range(self.stock_dim):
            if action[i] > 0:  # Buy
                shares_to_buy = min(self.hmax, self.current_cash // self.data["close"])
                self.current_cash -= shares_to_buy * self.data["close"] * (1 + self.buy_cost_pct)
                self.num_stock_shares[i] += shares_to_buy

            elif action[i] < 0:  # Sell
                shares_to_sell = min(self.hmax, self.num_stock_shares[i])
                self.current_cash += shares_to_sell * self.data["close"] * (1 - self.sell_cost_pct)
                self.num_stock_shares[i] -= shares_to_sell

    def _calculate_reward(self):
        """Calculates reward based on portfolio value"""
        total_assets = self.current_cash + np.sum(self.num_stock_shares * self.data["close"])
        reward = total_assets - self.initial_amount
        return reward * self.reward_scaling

    def _get_observation(self):
        """Returns the current state representation"""
        stock_data = [self.data[indicator] for indicator in self.tech_indicator_list]
        return np.array([self.current_cash] + list(self.num_stock_shares) + stock_data)

    def render(self):
        """Prints the environment state"""
        print(f"Day: {self.day}, Cash: {self.current_cash}, Holdings: {self.num_stock_shares}")


/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [43]:
import pandas as pd

# Example stock data (replace with real dataset)
data = {
    "date": pd.date_range(start="2023-01-01", periods=10, freq="D"),
    "close": [100, 102, 101, 105, 107, 110, 108, 109, 112, 115],
    "macd": np.random.randn(10),
    "rsi": np.random.randn(10),
    "cci": np.random.randn(10),
    "adx": np.random.randn(10),
    "vix": np.random.randn(10),
}
df = pd.DataFrame(data)

# Initialize Environment
env = StockTradingRiskEnv(
    df=df,
    stock_dim=1,
    hmax=10,
    initial_amount=1000,
    num_stock_shares=[0],
    buy_cost_pct=0.001,
    sell_cost_pct=0.001,
    reward_scaling=1e-4,
    state_space=6,
    action_space=1,
    tech_indicator_list=["macd", "rsi", "cci", "adx", "vix"]
)

# Test Environment
obs = env.reset()
done = False

while not done:
    action = np.random.uniform(-1, 1, size=(1,))  # Random buy/sell action
    obs, reward, done, _ = env.step(action)
    env.render()


Day: 1, Cash: 1000.0, Holdings: [0.]
Day: 2, Cash: 1000.0, Holdings: [0.]
Day: 3, Cash: 90.09100000000012, Holdings: [9.]
Day: 4, Cash: 90.09100000000012, Holdings: [9.]
Day: 5, Cash: 1052.1280000000002, Holdings: [0.]
Day: 6, Cash: 1052.1280000000002, Holdings: [0.]
Day: 7, Cash: 79.15600000000029, Holdings: [9.]
Day: 8, Cash: 79.15600000000029, Holdings: [9.]
Day: 9, Cash: 1086.1480000000001, Holdings: [0.]


**Train PPO Model in the StockTradingRiskEnv**

In [44]:
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

# Define technical indicators
tech_indicators = ["macd", "rsi", "returns", "volatility", "value_at_risk"]

# Initialize the StockTradingRiskEnv
env = StockTradingRiskEnv(
    df=df,
    stock_dim=1,
    hmax=100,
    initial_amount=100000,
    num_stock_shares=[0],
    buy_cost_pct=0.001,
    sell_cost_pct=0.001,
    reward_scaling=1e-4,
    state_space=len(tech_indicators) + 2,  # Cash balance + stock holdings
    action_space=1,
    tech_indicator_list=tech_indicators
)


In [45]:
!pip install 'shimmy>=2.0' # Install shimmy to bridge the gap between OpenAI Gym and Gymnasium

In [46]:
# Wrap in a Dummy Vectorized Environment (Stable-Baselines3 requirement)
vec_env = DummyVecEnv([lambda: env])

/usr/local/lib/python3.11/dist-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


In [47]:
tech_indicators = ["macd", "rsi", "returns", "volatility", "value_at_risk"]

In [ ]:
# Initialize PPO model
ppo_model = PPO("MlpPolicy", vec_env, verbose=1, learning_rate=0.0003, batch_size=64)

# Train PPO model
ppo_model.learn(total_timesteps=50000)  # Adjust timesteps as needed

# Save the trained model
ppo_model.save("ppo_stock_trading_risk")
print("PPO model training completed and saved!")
